# Deformable DETR for Cell Detection
Based on: https://github.com/fundamentalvision/Deformable-DETR

**Important**: Before running, compile MSDeformAttn CUDA operations:
```bash
cd models/ops
sh make.sh
# or: python setup.py build install
```

In [ ]:
# [실행순서 1] Cell 1: Imports and Setup (from GitHub)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models import resnet50
import numpy as np
import cv2
import os
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
import glob
import json
import math
import copy
from typing import Optional, List

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 6 클래스 cell type 정의 (HnE 데이터)
class_names = {
    0: "Neutrophil",
    1: "Epithelial",
    2: "Lymphocyte",
    3: "Plasma",
    4: "Eosinophil",
    5: "Connective tissue"
}

num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
print(f"Classes: {list(class_names.values())}")

In [ ]:
# [실행순서 2] Cell 2: Utility Classes (from GitHub - util/misc.py)

class NestedTensor:
    """Tensor with mask for batch padding"""
    def __init__(self, tensors, mask):
        self.tensors = tensors
        self.mask = mask
        
    def decompose(self):
        return self.tensors, self.mask
    
    def to(self, device):
        cast_tensor = self.tensors.to(device)
        cast_mask = self.mask.to(device) if self.mask is not None else None
        return NestedTensor(cast_tensor, cast_mask)

def nested_tensor_from_tensor_list(tensor_list):
    """Create NestedTensor from list of tensors (batch)"""
    if tensor_list[0].ndim == 3:  # [C, H, W]
        # Get max sizes
        max_size = tuple(max(s) for s in zip(*[img.shape for img in tensor_list]))
        batch_shape = (len(tensor_list),) + max_size
        b, c, h, w = batch_shape
        dtype = tensor_list[0].dtype
        device = tensor_list[0].device
        tensor = torch.zeros(batch_shape, dtype=dtype, device=device)
        mask = torch.ones((b, h, w), dtype=torch.bool, device=device)
        
        for img, pad_img, m in zip(tensor_list, tensor, mask):
            pad_img[: img.shape[0], : img.shape[1], : img.shape[2]].copy_(img)
            m[: img.shape[1], :img.shape[2]] = False
    else:
        raise ValueError('Not supported')
    return NestedTensor(tensor, mask)

def inverse_sigmoid(x, eps=1e-5):
    """Inverse of sigmoid function"""
    x = x.clamp(min=0, max=1)
    x1 = x.clamp(min=eps)
    x2 = (1 - x).clamp(min=eps)
    return torch.log(x1/x2)

print("✅ Utility classes loaded")

In [ ]:
# [실행순서 3] Cell 3: Corrected DETR-style Model for Point Detection
# Based on: https://github.com/facebookresearch/detr (original DETR)
# Note: Full Deformable DETR requires MSDeformAttn CUDA ops

class PositionEmbeddingSine(nn.Module):
    """
    Positional encoding using sine/cosine (from DETR GitHub)
    This is ESSENTIAL for transformer models!
    """
    def __init__(self, num_pos_feats=128, temperature=10000, normalize=True, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor_list):
        """Generate 2D positional encoding"""
        x = tensor_list.tensors
        mask = tensor_list.mask
        assert mask is not None
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos


class MLP(nn.Module):
    """Multi-Layer Perceptron (from DETR GitHub)"""
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(nn.Linear(n, k) for n, k in zip([input_dim] + h, h + [output_dim]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        return x


class DETR_PointDetection(nn.Module):
    """
    DETR for Point Detection (Corrected version)
    Based on official DETR: https://github.com/facebookresearch/detr
    
    Key differences from broken implementation:
    1. Uses separate encoder and decoder (not combined transformer)
    2. Adds positional encoding (ESSENTIAL!)
    3. Proper query embedding usage
    4. Correct loss computation
    """
    def __init__(self, num_classes=6, num_queries=300, hidden_dim=256, nheads=8,
                 num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.num_queries = num_queries
        self.hidden_dim = hidden_dim
        
        # Backbone: ResNet50 (from DETR)
        backbone = resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        
        # Reduce backbone channels to hidden_dim
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        
        # Positional encoding (ESSENTIAL - was missing!)
        self.position_embedding = PositionEmbeddingSine(hidden_dim // 2, normalize=True)
        
        # Transformer Encoder (processes image features)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nheads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        
        # Transformer Decoder (processes queries)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim,
            nhead=nheads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        
        # Learnable query embeddings (CORRECTED - not zeros!)
        self.query_embed = nn.Embedding(num_queries, hidden_dim)
        
        # Prediction heads
        self.class_embed = nn.Linear(hidden_dim, num_classes)
        self.point_embed = MLP(hidden_dim, hidden_dim, 2, 3)
        
        # Initialize weights
        self._reset_parameters()
    
    def _reset_parameters(self):
        """Initialize parameters (from DETR GitHub)"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        
        # Bias initialization for better initial predictions
        prior_prob = 0.01
        bias_value = -math.log((1 - prior_prob) / prior_prob)
        self.class_embed.bias.data = torch.ones(num_classes) * bias_value
        nn.init.constant_(self.point_embed.layers[-1].weight.data, 0)
        nn.init.constant_(self.point_embed.layers[-1].bias.data, 0)
    
    def forward(self, samples):
        """
        Args:
            samples: NestedTensor with images
        Returns:
            dict with pred_logits and pred_points
        """
        if not isinstance(samples, NestedTensor):
            samples = nested_tensor_from_tensor_list(samples)
        
        # Extract features from backbone
        features = self.backbone(samples.tensors)  # [B, 2048, H/32, W/32]
        features = self.input_proj(features)  # [B, hidden_dim, H, W]
        
        # Get positional encoding (ESSENTIAL!)
        mask = F.interpolate(samples.mask[None].float(), size=features.shape[-2:]).to(torch.bool)[0]
        pos_embed = self.position_embedding(NestedTensor(features, mask))  # [B, hidden_dim, H, W]
        
        # Flatten spatial dimensions for transformer
        bs, c, h, w = features.shape
        src = features.flatten(2).permute(0, 2, 1)  # [B, H*W, hidden_dim]
        pos_embed = pos_embed.flatten(2).permute(0, 2, 1)  # [B, H*W, hidden_dim]
        mask_flat = mask.flatten(1)  # [B, H*W]
        
        # Add positional encoding to features
        src = src + pos_embed
        
        # Transformer Encoder (encode image features)
        memory = self.encoder(src, src_key_padding_mask=mask_flat)  # [B, H*W, hidden_dim]
        
        # Query embeddings
        query_embed = self.query_embed.weight.unsqueeze(0).repeat(bs, 1, 1)  # [B, num_queries, hidden_dim]
        
        # Transformer Decoder (decode queries using image features)
        tgt = torch.zeros_like(query_embed)  # Initial query features
        hs = self.decoder(tgt, memory, memory_key_padding_mask=mask_flat)  # [B, num_queries, hidden_dim]
        
        # Predictions
        outputs_class = self.class_embed(hs)  # [B, num_queries, num_classes]
        outputs_coord = self.point_embed(hs).sigmoid()  # [B, num_queries, 2]
        
        out = {'pred_logits': outputs_class, 'pred_points': outputs_coord}
        return out


print("✅ DETR Point Detection model loaded (CORRECTED)")
print("   Key fixes:")
print("   1. ✅ Separate Encoder/Decoder (not combined transformer)")
print("   2. ✅ Positional encoding added (was missing!)")
print("   3. ✅ Proper query embedding usage")
print("   4. ✅ Correct forward pass flow")
print()
print("⚠️  Note: This is standard DETR, not Deformable DETR")
print("   For full Deformable DETR, you need MSDeformAttn CUDA ops")


In [ ]:
# [실행순서 4] Cell 4: Hungarian Matcher for Point Detection  
# Based on: https://github.com/fundamentalvision/Deformable-DETR/blob/main/models/matcher.py

class HungarianMatcherPoints(nn.Module):
    """
    Hungarian Matcher adapted for point detection (from GitHub)
    - Removed GIoU cost (no boxes)
    - Uses L2 distance for points
    """
    def __init__(self, cost_class=2.0, cost_point=5.0, focal_alpha=0.25):
        super().__init__()
        self.cost_class = cost_class
        self.cost_point = cost_point
        self.focal_alpha = focal_alpha
    
    @torch.no_grad()
    def forward(self, outputs, targets):
        """
        Args:
            outputs: dict with pred_logits [B, num_queries, num_classes] and pred_points [B, num_queries, 2]
            targets: list of dicts, each with 'labels' and 'points'
        Returns:
            list of (index_i, index_j) tuples for each batch
        """
        bs, num_queries = outputs["pred_logits"].shape[:2]
        
        # Flatten batch dimension
        out_prob = outputs["pred_logits"].flatten(0, 1).sigmoid()  # [B*num_queries, num_classes]
        out_points = outputs["pred_points"].flatten(0, 1)  # [B*num_queries, 2]
        
        # Concatenate target labels and points
        tgt_ids = torch.cat([v["labels"] for v in targets])
        tgt_points = torch.cat([v["points"] for v in targets])
        
        # Classification cost (Focal loss cost from GitHub)
        alpha = self.focal_alpha
        gamma = 2.0
        neg_cost_class = (1 - alpha) * (out_prob ** gamma) * (-(1 - out_prob + 1e-8).log())
        pos_cost_class = alpha * ((1 - out_prob) ** gamma) * (-(out_prob + 1e-8).log())
        cost_class = pos_cost_class[:, tgt_ids] - neg_cost_class[:, tgt_ids]
        
        # Point distance cost (L2)
        cost_point = torch.cdist(out_points, tgt_points, p=2)
        
        # Final cost matrix
        C = self.cost_point * cost_point + self.cost_class * cost_class
        C = C.view(bs, num_queries, -1).cpu()
        
        sizes = [len(v["points"]) for v in targets]
        indices = [linear_sum_assignment(c[i]) for i, c in enumerate(C.split(sizes, -1))]
        return [(torch.as_tensor(i, dtype=torch.int64), torch.as_tensor(j, dtype=torch.int64)) for i, j in indices]

print("✅ Hungarian Matcher loaded")

In [ ]:
# [실행순서 5] Cell 5: SetCriterion Loss for Point Detection
# Based on: https://github.com/fundamentalvision/Deformable-DETR/blob/main/models/deformable_detr.py

def sigmoid_focal_loss(inputs, targets, num_boxes, alpha=0.25, gamma=2):
    """
    Focal loss (from GitHub)
    Args:
        inputs: [N, num_classes]
        targets: [N, num_classes] (one-hot)
        num_boxes: normalizer
    """
    prob = inputs.sigmoid()
    ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
    p_t = prob * targets + (1 - prob) * (1 - targets)
    loss = ce_loss * ((1 - p_t) ** gamma)
    
    if alpha >= 0:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss
    
    return loss.mean(1).sum() / num_boxes


class SetCriterionPoints(nn.Module):
    """
    Loss computation for point detection (adapted from GitHub)
    - Focal loss for classification
    - L1 loss for points
    - No GIoU (removed from original)
    """
    def __init__(self, num_classes, matcher, weight_dict, focal_alpha=0.25):
        super().__init__()
        self.num_classes = num_classes
        self.matcher = matcher
        self.weight_dict = weight_dict
        self.focal_alpha = focal_alpha
    
    def loss_labels(self, outputs, targets, indices, num_boxes):
        """Classification loss (Focal Loss from GitHub)"""
        assert 'pred_logits' in outputs
        src_logits = outputs['pred_logits']
        
        idx = self._get_src_permutation_idx(indices)
        target_classes_o = torch.cat([t["labels"][J] for t, (_, J) in zip(targets, indices)])
        target_classes = torch.full(src_logits.shape[:2], self.num_classes,
                                    dtype=torch.int64, device=src_logits.device)
        target_classes[idx] = target_classes_o
        
        target_classes_onehot = torch.zeros([src_logits.shape[0], src_logits.shape[1], src_logits.shape[2] + 1],
                                            dtype=src_logits.dtype, layout=src_logits.layout, device=src_logits.device)
        target_classes_onehot.scatter_(2, target_classes.unsqueeze(-1), 1)
        target_classes_onehot = target_classes_onehot[:,:,:-1]
        
        loss_ce = sigmoid_focal_loss(src_logits, target_classes_onehot, num_boxes, 
                                     alpha=self.focal_alpha, gamma=2) * src_logits.shape[1]
        losses = {'loss_ce': loss_ce}
        return losses
    
    def loss_points(self, outputs, targets, indices, num_boxes):
        """Point regression loss (L1 from GitHub, adapted for points)"""
        assert 'pred_points' in outputs
        idx = self._get_src_permutation_idx(indices)
        src_points = outputs['pred_points'][idx]
        target_points = torch.cat([t['points'][i] for t, (_, i) in zip(targets, indices)], dim=0)
        
        loss_point = F.l1_loss(src_points, target_points, reduction='none')
        losses = {}
        losses['loss_point'] = loss_point.sum() / num_boxes
        return losses
    
    @torch.no_grad()
    def loss_cardinality(self, outputs, targets, indices, num_boxes):
        """Cardinality error (from GitHub) - for logging only"""
        pred_logits = outputs['pred_logits']
        device = pred_logits.device
        tgt_lengths = torch.as_tensor([len(v["labels"]) for v in targets], device=device)
        card_pred = (pred_logits.argmax(-1) != pred_logits.shape[-1] - 1).sum(1)
        card_err = F.l1_loss(card_pred.float(), tgt_lengths.float())
        losses = {'cardinality_error': card_err}
        return losses
    
    def _get_src_permutation_idx(self, indices):
        batch_idx = torch.cat([torch.full_like(src, i) for i, (src, _) in enumerate(indices)])
        src_idx = torch.cat([src for (src, _) in indices])
        return batch_idx, src_idx
    
    def _get_tgt_permutation_idx(self, indices):
        batch_idx = torch.cat([torch.full_like(tgt, i) for i, (_, tgt) in enumerate(indices)])
        tgt_idx = torch.cat([tgt for (_, tgt) in indices])
        return batch_idx, tgt_idx
    
    def forward(self, outputs, targets):
        """
        Args:
            outputs: dict with pred_logits and pred_points
            targets: list of dicts with labels and points
        """
        # Retrieve matching
        indices = self.matcher(outputs, targets)
        
        # Number of points
        num_boxes = sum(len(t["labels"]) for t in targets)
        num_boxes = torch.as_tensor([num_boxes], dtype=torch.float, device=next(iter(outputs.values())).device)
        num_boxes = torch.clamp(num_boxes, min=1).item()
        
        # Compute losses
        losses = {}
        losses.update(self.loss_labels(outputs, targets, indices, num_boxes))
        losses.update(self.loss_points(outputs, targets, indices, num_boxes))
        losses.update(self.loss_cardinality(outputs, targets, indices, num_boxes))
        
        return losses

print("✅ SetCriterion loss loaded")

In [ ]:
# [실행순서 6] Cell 6: Data Loading (same as p2p_train.ipynb)
input_size = 512
label_dir = '../../data/HnE_cell_detect/total_data/labels/'
image_dir = '../../data/HnE_cell_detect/total_data/images/'

label_files = sorted(glob.glob(os.path.join(label_dir, '*.json')))

image_filenames = []
labels = []

print("📂 Loading labels...")
for i in tqdm(range(len(label_files))):
    label_file = label_files[i]
    
    with open(label_file) as f:
        data_json = json.load(f)
    
    img_path = os.path.join(image_dir, data_json['file_name'])
    
    if os.path.exists(img_path):
        image_filenames.append(img_path)
        
        centers = []
        classes = []
        
        # JSON 형식: data_json["cordinates"] = [[class_id, y, x, h, w], ...]
        for coord in data_json["cordinates"]:
            if len(coord) < 5:
                continue
                
            class_id = int(coord[0]) - 1  # HnE 데이터는 1부터 시작하므로 -1
            y = coord[1]
            x = coord[2]
            h = coord[3]
            w = coord[4]
            
            # 너무 큰 박스 제외
            if h > 50 or w > 50:
                continue
            
            # 중심점 좌표 (픽셀 단위)
            centers.append([x+w//2, y+h//2])
            classes.append(class_id)
        
        if len(centers) > 0:
            labels.append({
                'points': np.array(centers, dtype=np.float32),
                'classes': np.array(classes, dtype=np.int64)
            })
        else:
            image_filenames.pop()

print(f"✅ Loaded {len(image_filenames)} images with labels")

print("\n📷 Loading images...")
images = []
for i in tqdm(range(len(image_filenames))):
    image = cv2.imread(image_filenames[i])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    images.append(image)

print(f"✅ Loaded {len(images)} images")
print(f"  Image shape: {images[0].shape}")
print(f"  Label example - Points: {labels[0]['points'].shape}, Classes: {labels[0]['classes'].shape}")

In [ ]:
# [실행순서 7] Cell 7: Dataset Class with HnE Augmentation
class DETRCellDataset(Dataset):
    """
    Dataset for DETR-style models with HnE-specific augmentation
    병리 이미지 스캐너/병원별 염색 차이에 robust하게 동작
    """
    def __init__(self, images, labels, input_size=512, augment=True):
        self.images = images
        self.labels = labels
        self.input_size = input_size
        self.augment = augment
    
    def __len__(self):
        return len(self.images)
    
    def apply_color_augmentation(self, image):
        """
        HnE-specific color augmentation for scanner/staining variation
        """
        # Brightness adjustment (±20%)
        if random.random() < 0.5:
            brightness_factor = random.uniform(0.8, 1.2)
            image = np.clip(image * brightness_factor, 0, 255).astype(np.uint8)
        
        # Contrast adjustment (±20%)
        if random.random() < 0.5:
            contrast_factor = random.uniform(0.8, 1.2)
            mean = image.mean()
            image = np.clip((image - mean) * contrast_factor + mean, 0, 255).astype(np.uint8)
        
        # Hue shift (stain variation)
        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            hue_shift = random.uniform(-10, 10)
            hsv[:, :, 0] = np.clip(hsv[:, :, 0] + hue_shift, 0, 179)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        
        # Saturation adjustment (stain intensity)
        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            saturation_factor = random.uniform(0.7, 1.3)
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * saturation_factor, 0, 255)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        
        # Gaussian noise (scanner noise)
        if random.random() < 0.3:
            noise_std = random.uniform(0, 5)
            noise = np.random.normal(0, noise_std, image.shape)
            image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        
        return image
    
    def __getitem__(self, idx):
        image = self.images[idx].copy()
        points = self.labels[idx]['points'].copy()
        classes = self.labels[idx]['classes'].copy()
        
        h, w = image.shape[:2]
        
        # Resize to input_size
        image_resized = cv2.resize(image, (self.input_size, self.input_size))
        scale_x = self.input_size / w
        scale_y = self.input_size / h
        points_resized = points * np.array([scale_x, scale_y])
        
        # Augmentation
        if self.augment:
            # Horizontal flip
            if random.random() > 0.5:
                image_resized = np.fliplr(image_resized).copy()
                points_resized[:, 0] = self.input_size - points_resized[:, 0]
            
            # Vertical flip
            if random.random() > 0.5:
                image_resized = np.flipud(image_resized).copy()
                points_resized[:, 1] = self.input_size - points_resized[:, 1]
            
            # Color augmentation (HnE-specific)
            image_resized = self.apply_color_augmentation(image_resized)
        
        # Normalize points to [0, 1]
        points_normalized = points_resized / self.input_size
        
        # Convert to tensor
        image_tensor = torch.from_numpy(image_resized).permute(2, 0, 1).float() / 255.0
        
        # Create target dict
        target = {
            'points': torch.from_numpy(points_normalized).float(),
            'labels': torch.from_numpy(classes).long()
        }
        
        return image_tensor, target


def collate_fn(batch):
    """Custom collate function for DETR"""
    images, targets = list(zip(*batch))
    images = nested_tensor_from_tensor_list(images)
    return images, targets


# Split data
from sklearn.model_selection import train_test_split
train_images, val_images, train_labels, val_labels = train_test_split(
    images, labels, test_size=0.2, random_state=42
)

train_dataset = DETRCellDataset(train_images, train_labels, input_size=input_size, augment=True)
val_dataset = DETRCellDataset(val_images, val_labels, input_size=input_size, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=4)

print(f"✅ Dataset created with HnE augmentation")
print(f"  Train: {len(train_dataset)} images (augmented)")
print(f"  Val: {len(val_dataset)} images (original)")
print()
print(f"🎨 HnE augmentations enabled:")
print(f"  - Geometric: Horizontal/Vertical flip")
print(f"  - Brightness: ±20%")
print(f"  - Contrast: ±20%")
print(f"  - Hue: ±10° (stain color)")
print(f"  - Saturation: ±30% (stain intensity)")
print(f"  - Gaussian noise: 0-5 std")


In [ ]:
# [실행순서 8] Cell 8: Training Setup (Corrected - from DETR GitHub)

# Model (CORRECTED)
model = DETR_PointDetection(
    num_classes=num_classes,
    num_queries=100,  # REDUCED from 300 (original DETR uses 100)
    hidden_dim=256,  # From DETR
    nheads=8,  # From DETR
    num_encoder_layers=6,  # From DETR
    num_decoder_layers=6,  # From DETR
    dim_feedforward=2048,  # From DETR
    dropout=0.1  # From DETR
).to(device)

# Matcher (same as before)
matcher = HungarianMatcherPoints(
    cost_class=1.0,  # DETR default (was 2.0)
    cost_point=5.0,  # bbox cost coefficient
    focal_alpha=0.25
)

# Loss weights (from DETR GitHub - corrected)
weight_dict = {
    'loss_ce': 1.0,  # DETR uses 1.0 for classification (was 2.0)
    'loss_point': 5.0  # DETR uses 5.0 for bbox (adapted for points)
}

criterion = SetCriterionPoints(
    num_classes=num_classes,
    matcher=matcher,
    weight_dict=weight_dict,
    focal_alpha=0.25
).to(device)

# Optimizer (from DETR GitHub - corrected)
# DETR doesn't use different LR for backbone
param_dicts = [
    {
        "params": [p for n, p in model.named_parameters() 
                  if "backbone" not in n and p.requires_grad]
    },
    {
        "params": [p for n, p in model.named_parameters() 
                  if "backbone" in n and p.requires_grad],
        "lr": 1e-5,  # Lower LR for pretrained backbone
    },
]
optimizer = torch.optim.AdamW(param_dicts, lr=1e-4, weight_decay=1e-4)

# Learning rate scheduler (from DETR - step decay)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=200, gamma=0.1)

print(f"✅ Training setup complete (CORRECTED)")
print(f"  Model: DETR Point Detection")
print(f"  Num queries: 100 (reduced from 300)")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print()
print(f"  Key improvements:")
print(f"  1. ✅ Positional encoding enabled")
print(f"  2. ✅ Proper encoder-decoder architecture")
print(f"  3. ✅ DETR-aligned hyperparameters")
print(f"  4. ✅ Reduced num_queries for efficiency")


In [ ]:
# [실행순서 10] Cell 10: Evaluation and Visualization

def visualize_predictions(model, dataset,save_path=None, num_samples=5, conf_threshold=0.5):
    """Visualize model predictions"""
    model.eval()
    
    # Create legend handles for classes
    legend_elements = []
    for cls_id, cls_name in class_names.items():
        color = plt.cm.tab10(cls_id)
        legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', 
                                         markerfacecolor=color, markersize=10, 
                                         label=cls_name, markeredgecolor='white', markeredgewidth=1))
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    with torch.no_grad():
        for i in range(num_samples):
            # Get sample
            image_tensor, target = dataset[i]
            image_batch = nested_tensor_from_tensor_list([image_tensor])
            image_batch = image_batch.to(device)
            
            # Predict
            outputs = model(image_batch)
            
            # Get predictions
            logits = outputs['pred_logits'][0]  # [num_queries, num_classes]
            points = outputs['pred_points'][0]  # [num_queries, 2]
            
            probs = logits.sigmoid()
            max_probs, pred_classes = probs.max(dim=-1)
            
            # Filter by confidence
            keep = max_probs > conf_threshold
            pred_points = points[keep].cpu().numpy()
            pred_classes_filtered = pred_classes[keep].cpu().numpy()
            pred_probs = max_probs[keep].cpu().numpy()
            
            # Prepare image for visualization
            image_np = (image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            
            # Ground truth
            gt_points = target['points'].numpy() * input_size
            gt_classes = target['labels'].numpy()
            
            # Plot original image
            axes[i, 0].imshow(image_np)
            axes[i, 0].set_title(f"Original Image {i+1}")
            axes[i, 0].axis('off')
            
            # Plot with ground truth
            axes[i, 1].imshow(image_np)
            for pt, cls in zip(gt_points, gt_classes):
                color = plt.cm.tab10(cls)
                axes[i, 1].plot(pt[0], pt[1], 'o', color=color, markersize=4, markeredgecolor='white', markeredgewidth=1)
            axes[i, 1].set_title(f"Ground Truth ({len(gt_points)} cells)")
            axes[i, 1].legend(handles=legend_elements, loc='upper right', fontsize=8, framealpha=0.8)
            axes[i, 1].axis('off')
            
            # Plot with predictions
            axes[i, 2].imshow(image_np)
            pred_points_scaled = pred_points * input_size
            for pt, cls, prob in zip(pred_points_scaled, pred_classes_filtered, pred_probs):
                color = plt.cm.tab10(cls)
                axes[i, 2].plot(pt[0], pt[1], 'o', color=color, markersize=4, markeredgewidth=1)
            axes[i, 2].set_title(f"Predictions ({len(pred_points)} cells, conf>{conf_threshold})")
            axes[i, 2].legend(handles=legend_elements, loc='upper right', fontsize=8, framealpha=0.8)
            axes[i, 2].axis('off')
    
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path)
    else:
        plt.show()

print("✅ Visualization function loaded")

In [ ]:
# [실행순서 9] Cell 9: Training Loop (adapted from GitHub engine.py)

num_epochs = 300
best_val_f1 = 0.0  # Changed from best_val_loss
save_dir = '../../model/HnE_cell_detection/Deformable_detr'
os.makedirs(save_dir, exist_ok=True)

# Training history
history = {
    'train_loss': [],
    'train_loss_ce': [],
    'train_loss_point': [],
    'val_loss': [],
    'val_loss_ce': [],
    'val_loss_point': [],
    'val_precision': [],
    'val_recall': [],
    'val_f1': [],
    'val_class_acc': []
}

print("🚀 Starting training...")
for epoch in range(num_epochs):
    # Training
    model.train()
    criterion.train()
    
    train_loss = 0
    train_loss_ce = 0
    train_loss_point = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for samples, targets in pbar:
        samples = samples.to(device)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward
        outputs = model(samples)
        loss_dict = criterion(outputs, targets)
        
        # Weighted sum of losses
        losses = sum(loss_dict[k] * weight_dict[k] for k in loss_dict.keys() if k in weight_dict)
        
        # Backward
        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)  # From GitHub
        optimizer.step()
        
        # Accumulate losses
        train_loss += losses.item()
        train_loss_ce += loss_dict['loss_ce'].item()
        train_loss_point += loss_dict['loss_point'].item()
        
        pbar.set_postfix({
            'loss': f"{losses.item():.4f}",
            'ce': f"{loss_dict['loss_ce'].item():.4f}",
            'pt': f"{loss_dict['loss_point'].item():.4f}"
        })
    
    train_loss /= len(train_loader)
    train_loss_ce /= len(train_loader)
    train_loss_point /= len(train_loader)
    
    history['train_loss'].append(train_loss)
    history['train_loss_ce'].append(train_loss_ce)
    history['train_loss_point'].append(train_loss_point)
    
    # Validation
    model.eval()
    criterion.eval()
    
    val_loss = 0
    val_loss_ce = 0
    val_loss_point = 0
    
    # Metrics for F1 calculation
    val_precision_sum = 0
    val_recall_sum = 0
    val_class_correct = 0
    val_class_total = 0
    val_samples = 0
    
    with torch.no_grad():
        for samples, targets in val_loader:
            samples = samples.to(device)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            outputs = model(samples)
            loss_dict = criterion(outputs, targets)
            
            losses = sum(loss_dict[k] * weight_dict[k] for k in loss_dict.keys() if k in weight_dict)
            
            val_loss += losses.item()
            val_loss_ce += loss_dict['loss_ce'].item()
            val_loss_point += loss_dict['loss_point'].item()
            
            # Calculate precision, recall, F1
            batch_size = len(targets)
            for b in range(batch_size):
                # Get predictions
                logits = outputs['pred_logits'][b]  # [num_queries, num_classes]
                points = outputs['pred_points'][b]  # [num_queries, 2]
                
                probs = logits.sigmoid()
                max_probs, pred_classes = probs.max(dim=-1)
                
                # Filter by confidence threshold
                conf_threshold = 0.5
                keep = max_probs > conf_threshold
                
                if keep.sum() == 0:
                    val_precision_sum += 0
                    val_recall_sum += 0
                    val_samples += 1
                    continue
                
                pred_points_filtered = points[keep]
                pred_classes_filtered = pred_classes[keep]
                
                # Get ground truth
                gt_points = targets[b]['points']
                gt_classes = targets[b]['labels']
                
                if len(gt_points) == 0:
                    continue
                
                # Hungarian matching for evaluation
                point_cost = torch.cdist(pred_points_filtered, gt_points, p=2)
                pred_probs_filtered = probs[keep]
                class_cost = -pred_probs_filtered[:, gt_classes]
                cost_matrix = (point_cost + class_cost).detach().cpu().numpy()
                
                pred_idx, gt_idx = linear_sum_assignment(cost_matrix)
                
                # Calculate metrics
                recall = len(gt_idx) / len(gt_points)
                precision = len(pred_idx) / keep.sum().item()
                
                val_recall_sum += recall
                val_precision_sum += precision
                
                # Class accuracy
                matched_pred_classes = pred_classes_filtered[pred_idx]
                matched_gt_classes = gt_classes[gt_idx]
                correct = (matched_pred_classes == matched_gt_classes).sum().item()
                val_class_correct += correct
                val_class_total += len(gt_idx)
                
                val_samples += 1
    
    val_loss /= len(val_loader)
    val_loss_ce /= len(val_loader)
    val_loss_point /= len(val_loader)
    
    # Calculate average metrics
    val_precision = val_precision_sum / val_samples if val_samples > 0 else 0
    val_recall = val_recall_sum / val_samples if val_samples > 0 else 0
    val_f1 = 2 * val_precision * val_recall / (val_precision + val_recall + 1e-8)
    val_class_acc = val_class_correct / val_class_total if val_class_total > 0 else 0
    
    history['val_loss'].append(val_loss)
    history['val_loss_ce'].append(val_loss_ce)
    history['val_loss_point'].append(val_loss_point)
    history['val_precision'].append(val_precision)
    history['val_recall'].append(val_recall)
    history['val_f1'].append(val_f1)
    history['val_class_acc'].append(val_class_acc)
    
    # Update LR
    lr_scheduler.step()
    
    # Print epoch summary
    print(f"\n{'='*80}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} (CE: {train_loss_ce:.4f}, Point: {train_loss_point:.4f})")
    print(f"  Val Loss: {val_loss:.4f} (CE: {val_loss_ce:.4f}, Point: {val_loss_point:.4f})")
    print(f"  Val Class Acc: {val_class_acc:.4f}")
    print(f"  Val Recall: {val_recall:.4f}")
    print(f"  Val Precision: {val_precision:.4f}")
    print(f"  Val F1: {val_f1:.4f} ⭐")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"{'='*80}\n")
    
    # Save best model based on F1 score
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_class_acc': val_class_acc,
        }, os.path.join(save_dir, 'best_model.pth'))
        print(f"  🎉 Saved best model (val_f1: {val_f1:.4f})")
    
    # Save checkpoint every 100 epochs  
    if (epoch + 1) % 100 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_class_acc': val_class_acc,
        }, os.path.join(save_dir, f'checkpoint_epoch{epoch+1}.pth'))
        print(f"  💾 Saved checkpoint")
    
    # Visualize every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"  📊 Visualizing predictions...")
        visualize_predictions(model, val_dataset, save_path=os.path.join(save_dir, f'visualize_epoch_{epoch+1}.jpg'), num_samples=3, conf_threshold=0.3)

print("\n" + "="*80)
print("✅ Training complete!")
print(f"  Best Val F1: {best_val_f1:.4f}")
print(f"  Models saved to: {save_dir}")
print("="*80)


In [ ]:
visualize_predictions(model, val_dataset, num_samples=3, conf_threshold=0.3)

In [ ]:
# [실행순서 11] Cell 11: Plot Training History

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.title('Total Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(history['train_loss_ce'], label='Train')
plt.plot(history['val_loss_ce'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Classification Loss')
plt.title('Classification Loss (Focal)')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(history['train_loss_point'], label='Train')
plt.plot(history['val_loss_point'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Point Loss')
plt.title('Point Regression Loss (L1)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# [실행순서 12] Cell 12: Visualize Predictions on Validation Set

# Load best model
checkpoint = torch.load(os.path.join(save_dir, 'best_model.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"   Best val_loss: {checkpoint['val_loss']:.4f}")

# Visualize
visualize_predictions(model, val_dataset, num_samples=5, conf_threshold=0.3)